# Import các thư viện cần thiết

In [ ]:
 import torch
 from trl import SFTTrainer
 from datasets import load_dataset
 from unsloth import FastLanguageModel
 from unsloth import FastLanguageModel
 from transformers import GenerationConfig
 from transformers import TrainingArguments

In [ ]:
login(token=HUGGINGFACE_TOKEN)

# Khai báo mô hình

In [ ]:
MAX_SEQ_LENGTH = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "up_proj",
        "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Tải và tổ chức dữ liệu multiturn conservation

In [ ]:
dataset_name = "5CD-AI/Vietnamese-Multi-turn-Chat-Alpaca"
raw_dataset = load_dataset(dataset_name, split="train")

In [ ]:
SYS_INSTRUCT = "Bạn là một trợ lý AI thân thiện, hãy trả lời bằng tiếng Việt."

def convert_to_chat_format(conservations):
    messages = [{"role": "system", "content": SYS_INSTRUCT}]
    for msg in conservations:
        role = "user" if msg["from"] == "human" else "assistant"
        messages.append({"role": role, "content": msg["value"]})
    return messages

def format_prompt(example):
    messages = convert_to_chat_format(example["conservations"])
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenizer=False,
            add_generation_prompt=False
        )
    }

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )

dataset = raw_dataset.map(format_prompt, remove_columns=raw_dataset.column_names)
dataset = dataset.map(tokenize_function, batched=True)

# Huấn luyện mô hình

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    save_total_limit=4,
    logging_steps=20,
    output_dir="./checkpoint/llama3-1b-multi-conversation",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    save_strategy="steps",
    save_steps=50,
    report_to="none",
    remove_unused_columns=True,
    max_steps=400,
    bf16=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    "./checkpoint/llama3-1b-multi-conversation/checkpoint-400",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

# Lưu mô hình và inference

In [ ]:
model.push_to_hub_merged(
    "V1nk4n/Llama-3.2-1B-Instruct-Chat-sft",
    commit_message="Merge weights to push to hub",
)

tokenizer.push_to_hub)
    "V1nk4n/Llama-3.2-1B-Instruct-Chat-sft",
    commit_message="Push tokenizer to hub",
)

In [ ]:
generation_config = GenerationConfig(
    max_new_tokens=128,
    temperature=1.0,
    do_sample=False,
    num_return_sequences=1,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.3
)

prompt = [
     {"role": "system", "content": (
        "Bạn là một trợ lý AI thân thiện, "
        "hãy trả lời bằng tiếng Việt."
    )},

    {"role": "user", "content": (
        "Hãy chỉnh sửa câu này để ngắn gọn hơn mà không mất đi ý nghĩa: "
        "\"Trận đấu là một thất bại nặng nề "
        "mặc dù thực tế là cả đội đã tập luyện trong nhiều tuần.\""
    )},

    {"role": "assistant", "content": (
        "Nhiều tuần huấn luyện của đội đã dẫn đến một thất bại nặng nề."
    )},

    {"role": "user", "content": (
        "Bạn có thể đề xuất một số chiến lược mà "
        "nhóm có thể sử dụng để cải thiện hiệu suất "
        "của họ trong trận đấu tiếp theo không?"
    )}
]

chat_text = tokenizer.apply_chat_template(
    prompt,
    add_generation_prompt=True,
    tokenize=False
)

inputs = tokenizer(
    chat_text,
    return_tensors="pt"
).to("cuda:0")

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        generation_config=generation_config,
    )
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

if "assistant\n" in output_text:
    answer = output_text.split("assistant\n")[-1].strip()
else:
    answer = output_text.strip()

print("Assistant reply:", answer)